# SparkSession

In [ ]:
#pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MyFisrtsCSVLoad").getOrCreate()

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
df = spark.read.csv(
    path="ratings.csv",
    sep=",",
    header=True,
    quote='"',
    inferSchema=True,
    schema="userId INT, movieId INT, rating DOUBLE, timestamp INT"
)

In [ ]:
df.show(5)

df.printSchema()

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [ ]:
from pyspark.sql import functions as f

* `withColumnRenamed` : renombrar columnas

* `f.from_unixtime`: crea una nueva columna donde transforma el número en una fecha y hora elegible

In [ ]:
# f.from_unixtime
# f.to_timestamp

df = (
    df
    .withColumnRenamed("timestamp", "timestamp_unix")
    .withColumn("timestamp", f.from_unixtime("timestamp_unix"))
    .withColumn("timestamp", f.to_timestamp("timestamp")
    ))

df.show(5)
df.printSchema()

+------+-------+------+--------------+-------------------+
|userId|movieId|rating|timestamp_unix|          timestamp|
+------+-------+------+--------------+-------------------+
|     1|      1|   4.0|     964982703|2000-07-30 18:45:03|
|     1|      3|   4.0|     964981247|2000-07-30 18:20:47|
|     1|      6|   4.0|     964982224|2000-07-30 18:37:04|
|     1|     47|   5.0|     964983815|2000-07-30 19:03:35|
|     1|     50|   5.0|     964982931|2000-07-30 18:48:51|
+------+-------+------+--------------+-------------------+
only showing top 5 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp_unix: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)



* `drop`: eliminar columnas

In [ ]:
df.drop("timestamp_unix").show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-30 18:45:03|
|     1|      3|   4.0|2000-07-30 18:20:47|
|     1|      6|   4.0|2000-07-30 18:37:04|
|     1|     47|   5.0|2000-07-30 19:03:35|
|     1|     50|   5.0|2000-07-30 18:48:51|
+------+-------+------+-------------------+
only showing top 5 rows


In [ ]:
ratings = spark.read.csv(
    path="ratings.csv",
    sep=",",
    header=True,
    quote='"',
    schema="userId INT, movieId INT, rating DOUBLE, timestamp INT"
).withColumn("timestamp", f.to_timestamp(f.from_unixtime("timestamp")))

In [ ]:
ratings.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-30 18:45:03|
|     1|      3|   4.0|2000-07-30 18:20:47|
|     1|      6|   4.0|2000-07-30 18:37:04|
|     1|     47|   5.0|2000-07-30 19:03:35|
|     1|     50|   5.0|2000-07-30 18:48:51|
+------+-------+------+-------------------+
only showing top 5 rows


In [ ]:
ratings.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [ ]:
movies = (
    spark.read.csv(
        path = "movies.csv",
        sep =",",
        header = True,
        quote = '"',
        inferSchema= True,
    )
    )

In [ ]:
movies.show(15, truncate= False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
|6      |Heat (1995)                       |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                    |Comedy|Romance                             |
|8      |Tom and Huck (1995)               |Adventure|Children                         |
|9      |Sudden Death

In [ ]:
movies.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



* `where`: filtrar alguna condición

In [ ]:
movies.where("genres = 'Action'").show()

+-------+--------------------+------+
|movieId|               title|genres|
+-------+--------------------+------+
|      9| Sudden Death (1995)|Action|
|     71|    Fair Game (1995)|Action|
|    204|Under Siege 2: Da...|Action|
|    251|  Hunted, The (1995)|Action|
|    667|Bloodsport 2 (a.k...|Action|
|   1170|Best of the Best ...|Action|
|   1497|  Double Team (1997)|Action|
|   1599|        Steel (1997)|Action|
|   2196|    Knock Off (1998)|Action|
|   2534|    Avalanche (1978)|Action|
|   2817|Aces: Iron Eagle ...|Action|
|   2965|Omega Code, The (...|Action|
|   3283|Minnie and Moskow...|Action|
|   3444|   Bloodsport (1988)|Action|
|   3769|Thunderbolt and L...|Action|
|   4200|Double Impact (1991)|Action|
|   4387|Kiss of the Drago...|Action|
|   4441|Game of Death (1978)|Action|
|   4531|     Red Heat (1988)|Action|
|   4568|Best of the Best ...|Action|
+-------+--------------------+------+
only showing top 20 rows


* `f.col + where`: filtrar una columna con un argumento

In [ ]:
movies.where(f.col("genres") == "Action").show(5, False)
movies.where("genres = 'Action'").show(5, False)

+-------+-----------------------------------------------------------+------+
|movieId|title                                                      |genres|
+-------+-----------------------------------------------------------+------+
|9      |Sudden Death (1995)                                        |Action|
|71     |Fair Game (1995)                                           |Action|
|204    |Under Siege 2: Dark Territory (1995)                       |Action|
|251    |Hunted, The (1995)                                         |Action|
|667    |Bloodsport 2 (a.k.a. Bloodsport II: The Next Kumite) (1996)|Action|
+-------+-----------------------------------------------------------+------+
only showing top 5 rows
+-------+-----------------------------------------------------------+------+
|movieId|title                                                      |genres|
+-------+-----------------------------------------------------------+------+
|9      |Sudden Death (1995)                        

* `split`: separar los valores

* `explode` :  expande una columna que contiene una lista de multiples filas
* `select`: sleccionar algunas columnas de la consulta

In [ ]:
movie_genre = (
    movies
    .withColumn("genres_array", f.split("genres", "\|"))
    .withColumn("genre", f.explode("genres_array"))
    .select("movieId", "title", "genre")
    )
movie_genre.show(5)

<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:3: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_12007/3900294226.py:3: SyntaxWarning: invalid escape sequence '\|'
  .withColumn("genres_array", f.split("genres", "\|"))


+-------+----------------+---------+
|movieId|           title|    genre|
+-------+----------------+---------+
|      1|Toy Story (1995)|Adventure|
|      1|Toy Story (1995)|Animation|
|      1|Toy Story (1995)| Children|
|      1|Toy Story (1995)|   Comedy|
|      1|Toy Story (1995)|  Fantasy|
+-------+----------------+---------+
only showing top 5 rows


In [ ]:
movie_genre.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = false)



* `distinct`: cuantas categorías hay por variable

In [ ]:
available_genres = movie_genre.select("genre").distinct()
available_genres.show()

+------------------+
|             genre|
+------------------+
|             Crime|
|           Romance|
|          Thriller|
|         Adventure|
|             Drama|
|               War|
|       Documentary|
|           Fantasy|
|           Mystery|
|           Musical|
|         Animation|
|         Film-Noir|
|(no genres listed)|
|              IMAX|
|            Horror|
|           Western|
|            Comedy|
|          Children|
|            Action|
|            Sci-Fi|
+------------------+



In [ ]:
movies_without_genre = movies.where(f.col("genres") == "(no genres listed)")

In [ ]:
movies_without_genre.count()


34

In [ ]:
movies_without_genre.show()

+-------+--------------------+------------------+
|movieId|               title|            genres|
+-------+--------------------+------------------+
| 114335|   La cravate (1957)|(no genres listed)|
| 122888|      Ben-hur (2016)|(no genres listed)|
| 122896|Pirates of the Ca...|(no genres listed)|
| 129250|   Superfast! (2015)|(no genres listed)|
| 132084| Let It Be Me (1995)|(no genres listed)|
| 134861|Trevor Noah: Afri...|(no genres listed)|
| 141131|    Guardians (2016)|(no genres listed)|
| 141866|   Green Room (2015)|(no genres listed)|
| 142456|The Brand New Tes...|(no genres listed)|
| 143410|          Hyena Road|(no genres listed)|
| 147250|The Adventures of...|(no genres listed)|
| 149330|A Cosmic Christma...|(no genres listed)|
| 152037|  Grease Live (2016)|(no genres listed)|
| 155589|Noin 7 veljestä (...|(no genres listed)|
| 156605|            Paterson|(no genres listed)|
| 159161|Ali Wong: Baby Co...|(no genres listed)|
| 159779|A Midsummer Night...|(no genres listed)|


In [ ]:
from os import path
links = spark.read.csv(
    path = "links.csv",
    sep = ",",
    header = True,
    quote = '"',
    schema = "movieId INT, imdbId INT, tmdbId INT"
)

tags = spark.read.csv(
    path = "tags.csv",
    sep = ",",
    header = True,
    quote = '"',
    schema = "userId INT, movieId INT, tag STRING, timestamp INT"
).withColumn("timestamp", f.to_timestamp(f.from_unixtime("timestamp")))

In [ ]:
links.show(5)

+-------+------+------+
|movieId|imdbId|tmdbId|
+-------+------+------+
|      1|114709|   862|
|      2|113497|  8844|
|      3|113228| 15602|
|      4|114885| 31357|
|      5|113041| 11862|
+-------+------+------+
only showing top 5 rows


In [ ]:
links.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- imdbId: integer (nullable = true)
 |-- tmdbId: integer (nullable = true)



In [ ]:
tags.show()
tags.printSchema()

+------+-------+-----------------+-------------------+
|userId|movieId|              tag|          timestamp|
+------+-------+-----------------+-------------------+
|     2|  60756|            funny|2015-10-24 19:29:54|
|     2|  60756|  Highly quotable|2015-10-24 19:29:56|
|     2|  60756|     will ferrell|2015-10-24 19:29:52|
|     2|  89774|     Boxing story|2015-10-24 19:33:27|
|     2|  89774|              MMA|2015-10-24 19:33:20|
|     2|  89774|        Tom Hardy|2015-10-24 19:33:25|
|     2| 106782|            drugs|2015-10-24 19:30:54|
|     2| 106782|Leonardo DiCaprio|2015-10-24 19:30:51|
|     2| 106782|  Martin Scorsese|2015-10-24 19:30:56|
|     7|  48516|     way too long|2007-01-25 01:08:45|
|    18|    431|        Al Pacino|2016-05-01 21:39:25|
|    18|    431|         gangster|2016-05-01 21:39:09|
|    18|    431|            mafia|2016-05-01 21:39:15|
|    18|   1221|        Al Pacino|2016-04-26 19:35:06|
|    18|   1221|            Mafia|2016-04-26 19:35:03|
|    18|  

* `groupBy`: se utiliza para agrupar por categoría

In [ ]:
movies_per_genre = movie_genre.groupBy("genre").count()

In [ ]:
movies_per_genre.show()

+------------------+-----+
|             genre|count|
+------------------+-----+
|             Crime| 1199|
|           Romance| 1596|
|          Thriller| 1894|
|         Adventure| 1263|
|             Drama| 4361|
|               War|  382|
|       Documentary|  440|
|           Fantasy|  779|
|           Mystery|  573|
|           Musical|  334|
|         Animation|  611|
|         Film-Noir|   87|
|(no genres listed)|   34|
|              IMAX|  158|
|            Horror|  978|
|           Western|  167|
|            Comedy| 3756|
|          Children|  664|
|            Action| 1828|
|            Sci-Fi|  980|
+------------------+-----+



* `join`: se utiliza para union de tablas

In [ ]:
opinions = movies.join(tags, movies["movieId"] == tags["movieId"], "left")
opinions.show(10)

+-------+--------------------+--------------------+------+-------+----------------+-------------------+
|movieId|               title|              genres|userId|movieId|             tag|          timestamp|
+-------+--------------------+--------------------+------+-------+----------------+-------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|   567|      1|             fun|2018-05-02 18:33:33|
|      1|    Toy Story (1995)|Adventure|Animati...|   474|      1|           pixar|2006-01-14 02:47:05|
|      1|    Toy Story (1995)|Adventure|Animati...|   336|      1|           pixar|2006-02-04 09:36:04|
|      2|      Jumanji (1995)|Adventure|Childre...|   474|      2|            game|2006-01-16 01:39:12|
|      2|      Jumanji (1995)|Adventure|Childre...|    62|      2|  Robin Williams|2018-06-12 22:51:47|
|      2|      Jumanji (1995)|Adventure|Childre...|    62|      2|magic board game|2018-06-12 22:52:12|
|      2|      Jumanji (1995)|Adventure|Childre...|    62|      

In [ ]:
opinions = movies.join(tags, ["movieId"])

In [ ]:
opinions.select("movieId").show()

+-------+
|movieId|
+-------+
|      1|
|      1|
|      1|
|      2|
|      2|
|      2|
|      2|
|      3|
|      3|
|      5|
|      5|
|      7|
|     11|
|     11|
|     14|
|     14|
|     16|
|     17|
|     21|
|     22|
+-------+
only showing top 20 rows


In [ ]:
opinions = (
    movies.join(tags, ["movieId"], "inner")
    .select("userId", "movieId", "title", "tag", "timestamp")
)

In [ ]:
opinions.join(ratings, ["movieId", "userId"]).show()

+-------+------+--------------------+--------------------+----------------+-------------------+------+-------------------+
|movieId|userId|               title|              genres|             tag|          timestamp|rating|          timestamp|
+-------+------+--------------------+--------------------+----------------+-------------------+------+-------------------+
|      1|   567|    Toy Story (1995)|Adventure|Animati...|             fun|2018-05-02 18:33:33|   3.5|2018-05-02 18:33:21|
|      1|   474|    Toy Story (1995)|Adventure|Animati...|           pixar|2006-01-14 02:47:05|   4.0|2001-01-04 02:36:00|
|      1|   336|    Toy Story (1995)|Adventure|Animati...|           pixar|2006-02-04 09:36:04|   4.0|2005-07-24 17:48:49|
|      2|   474|      Jumanji (1995)|Adventure|Childre...|            game|2006-01-16 01:39:12|   3.0|2003-03-05 17:53:34|
|      2|    62|      Jumanji (1995)|Adventure|Childre...|  Robin Williams|2018-06-12 22:51:47|   4.0|2018-06-12 22:51:30|
|      2|    62|

In [ ]:
opinions.show()

+------+-------+--------------------+----------------+-------------------+
|userId|movieId|               title|             tag|          timestamp|
+------+-------+--------------------+----------------+-------------------+
|   567|      1|    Toy Story (1995)|             fun|2018-05-02 18:33:33|
|   474|      1|    Toy Story (1995)|           pixar|2006-01-14 02:47:05|
|   336|      1|    Toy Story (1995)|           pixar|2006-02-04 09:36:04|
|   474|      2|      Jumanji (1995)|            game|2006-01-16 01:39:12|
|    62|      2|      Jumanji (1995)|  Robin Williams|2018-06-12 22:51:47|
|    62|      2|      Jumanji (1995)|magic board game|2018-06-12 22:52:12|
|    62|      2|      Jumanji (1995)|         fantasy|2018-06-12 22:52:09|
|   289|      3|Grumpier Old Men ...|             old|2006-03-27 02:01:00|
|   289|      3|Grumpier Old Men ...|           moldy|2006-03-27 02:01:00|
|   474|      5|Father of the Bri...|          remake|2006-01-16 01:11:43|
|   474|      5|Father of

In [ ]:
opinions_ext = opinions.withColumnRenamed("timestamp", "tag_time").join(ratings, ["movieId", "userId"])

In [ ]:
opinions_ext.show()

In [ ]:
movies.count()

In [ ]:
opinions.count()

In [ ]:
opinions_ext.count()

In [ ]:
ratings.count()

In [ ]:
ratings.select("userId").distinct().count()

In [ ]:
tags.select("userId").distinct().count()

In [ ]:
tags.select("movieId").distinct().count()

* `agg`: realiza calculos matematicos sobre grupos de datos.
  * `+ GroupBy`: separar los datos por categorías

In [ ]:
ratings.groupBy("movieId").agg(
    f.count("*"),
    f.min("rating"),
    f.max("rating"),
    f.avg("rating"),
    f.min("timestamp"),
    f.max("timestamp")
).show()

+-------+--------+-----------+-----------+------------------+-------------------+-------------------+
|movieId|count(1)|min(rating)|max(rating)|       avg(rating)|     min(timestamp)|     max(timestamp)|
+-------+--------+-----------+-----------+------------------+-------------------+-------------------+
|   1580|     165|        0.5|        5.0| 3.487878787878788|1997-07-07 12:07:18|2018-07-22 13:30:52|
|   2366|      25|        1.5|        5.0|              3.64|1999-11-04 15:23:49|2018-02-20 10:20:35|
|   3175|      75|        1.0|        5.0|              3.58|1999-12-26 14:01:31|2018-06-25 05:07:19|
|   1088|      42|        1.0|        5.0| 3.369047619047619|1997-04-07 07:36:08|2018-01-17 01:52:47|
|  32460|       4|        3.5|        5.0|              4.25|2011-12-18 19:21:21|2017-04-21 20:12:30|
|  44022|      23|        1.0|        4.5| 3.217391304347826|2006-10-25 18:02:59|2018-03-07 07:38:56|
|  96488|       4|        4.0|        4.5|              4.25|2014-11-08 16:17:07|2

* `collect_set`: sirve para agrupar todos los valores de una columna en una lista de elemntos únicos, eliminando los duplicados.

In [ ]:
from pyspark.sql import functions as f

# Corrección de alias y ordenamiento
tags.groupBy("movieId").agg(
    f.collect_set("tag").alias("tags"),
    f.count("tag").alias("tag_count"),           # Se agregó el alias 'tag_count'
    f.collect_set("userId").alias("users"),
    f.count("userId").alias("user_count"),
    f.min("timestamp").alias("first_tagged_date"),
    f.max("timestamp").alias("last_tagged_date")
).sort(f.col("tag_count").desc()).show()          # Ahora sí encuentra 'tag_count'

+-------+--------------------+---------+--------------------+----------+-------------------+-------------------+
|movieId|                tags|tag_count|               users|user_count|  first_tagged_date|   last_tagged_date|
+-------+--------------------+---------+--------------------+----------+-------------------+-------------------+
|    296|[mobsters, Tarant...|      181|[103, 599, 424, 474]|       181|2006-01-14 02:16:41|2017-06-26 05:58:14|
|   2959|[schizophrenia, m...|       54|[435, 599, 424, 474]|        54|2006-01-14 02:17:14|2017-06-26 06:02:47|
|    924|[apes, aliens, sc...|       41|          [599, 474]|        41|2006-01-15 23:33:55|2017-06-26 06:00:27|
|    293|[humorous, hit me...|       35|     [166, 599, 474]|        35|2006-01-24 21:20:19|2017-06-26 05:49:52|
|   7361|[intelligent, cul...|       34|[193, 567, 424, 4...|        34|2006-01-14 02:30:08|2018-05-02 17:40:11|
|   1732|[satirical, Coen ...|       32|          [599, 474]|        32|2006-01-26 20:17:41|2017

In [ ]:
ratings.groupBy("userId").agg(
    f.collect_set("movieId").alias("movieIds"),
    f.count("*"),
    f.avg("rating"),
    f.min("rating"),
    f.max("rating")
).show()

+------+--------------------+--------+------------------+-----------+-----------+
|userId|            movieIds|count(1)|       avg(rating)|min(rating)|max(rating)|
+------+--------------------+--------+------------------+-----------+-----------+
|     1|[1024, 2048, 1, 1...|     232| 4.366379310344827|        1.0|        5.0|
|     2|[115713, 122882, ...|      29|3.9482758620689653|        2.0|        5.0|
|     3|[5764, 1093, 647,...|      39|2.4358974358974357|        0.5|        5.0|
|     4|[1025, 3079, 3591...|     216|3.5555555555555554|        1.0|        5.0|
|     5|[1, 515, 261, 265...|      44|3.6363636363636362|        1.0|        5.0|
|     6|[2, 3, 515, 4, 51...|     314|3.4936305732484074|        1.0|        5.0|
|     7|[34048, 1, 8961, ...|     152|3.2302631578947367|        0.5|        5.0|
|     8|[2, 454, 457, 10,...|      47| 3.574468085106383|        1.0|        5.0|
|     9|[3328, 5952, 4993...|      46| 3.260869565217391|        1.0|        5.0|
|    10|[5377, 7

# ALS model

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
spark = SparkSession.builder.appName("Chapter4-3").getOrCreate()

In [2]:
ratings = (spark.read.csv(
    path = "ratings.csv",
    sep = ",",
    header = True,
    quote = '"',
    schema = "userId INT, movieId INT, rating DOUBLE, timestamp INT"
)
.withColumn("timestamp", f.to_timestamp(f.from_unixtime("timestamp")))
.cache()
)


In [3]:
ratings = (spark.read.csv(
    path = "ratings.csv",
    sep = ",",
    header = True,
    quote = '"', # Corrected from '""' to '"'
    schema = "userId INT, movieId INT, rating DOUBLE, timestamp INT"
)
.withColumn("timestamp", f.to_timestamp(f.from_unixtime("timestamp")))
.cache()
)
ratings.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-30 18:45:03|
|     1|      3|   4.0|2000-07-30 18:20:47|
|     1|      6|   4.0|2000-07-30 18:37:04|
|     1|     47|   5.0|2000-07-30 19:03:35|
|     1|     50|   5.0|2000-07-30 18:48:51|
+------+-------+------+-------------------+
only showing top 5 rows


In [4]:
ratings.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [5]:
ratings.summary().show()

+-------+------------------+----------------+------------------+
|summary|            userId|         movieId|            rating|
+-------+------------------+----------------+------------------+
|  count|            100836|          100836|            100836|
|   mean|326.12756356856676|19435.2957177992| 3.501556983616962|
| stddev| 182.6184914635004|35530.9871987003|1.0425292390606342|
|    min|                 1|               1|               0.5|
|    25%|               177|            1199|               3.0|
|    50%|               325|            2991|               3.5|
|    75%|               477|            8092|               4.0|
|    max|               610|          193609|               5.0|
+-------+------------------+----------------+------------------+



In [6]:
from pyspark.ml.recommendation import	ALS

In [7]:
class pyspark.ml.recommendation. ALS(
  rank=10,
  maxIter=10,
  regParam=0.1,
  numUserBlocks=10,
  numItemBlocks=10,
  implicitPrefs=False,
  alpha=1.0,
  userCol="user",
  itemCol="item",
  seed=None,
  ratingCol="rating",
  nonnegative=False,
  checkpointInterval=10,
  intermediateStorageLevel="MEMORY_AND_DISK",
  finalStorageLevel="MEMORY_AND_DISK",
  coldStartStrategy="nan",

  )

SyntaxError: invalid syntax (3276659439.py, line 1)

In [8]:
ratings = ratings.drop("timestamp")
ratings.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)



ALS: modelo de predicciones. Algoritmo de Filtrado Colaborativo que anliza el comportamiento.

* userCol="userId": Le dice al modelo quién es el sujeto. Spark necesita una columna con números de ID de usuario.

* itemCol="movieId": Le dice qué es lo que se está evaluando (en este caso, películas). También deben ser números.

* ratingCol="rating": Es la "calificación" que el usuario le dio (ej. de 1 a 5 estrellas). Es lo que el modelo intentará predecir para las películas que el usuario aún no ha visto.

* .fit(ratings): Es el momento donde el modelo "estudia". Toma tu tabla de datos (ratings) y busca patrones matemáticos ocultos para aprender las preferencias.

In [9]:
model = (
    ALS(
        userCol="userId",
        itemCol="movieId",
        ratingCol="rating"
    ).fit(ratings)
)

In [10]:
type(model)

pyspark.ml.recommendation.ALSModel

In [11]:
predictions = model.transform(ratings)

In [12]:
type(predictions)

pyspark.sql.classic.dataframe.DataFrame

In [13]:
predictions.show(30, False)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|1     |1      |4.0   |4.5405154 |
|1     |3      |4.0   |4.159376  |
|1     |6      |4.0   |4.415881  |
|1     |47     |5.0   |4.5818343 |
|1     |50     |5.0   |4.725305  |
|1     |70     |3.0   |3.6900105 |
|1     |101    |5.0   |4.693769  |
|1     |110    |4.0   |4.652298  |
|1     |151    |5.0   |4.133419  |
|1     |157    |5.0   |4.0233865 |
|1     |163    |5.0   |4.202178  |
|1     |216    |5.0   |4.2709846 |
|1     |223    |3.0   |4.557542  |
|1     |231    |5.0   |3.9751112 |
|1     |235    |4.0   |3.8750792 |
|1     |260    |5.0   |4.945283  |
|1     |296    |3.0   |4.8429227 |
|1     |316    |3.0   |3.8688588 |
|1     |333    |5.0   |4.3957596 |
|1     |349    |4.0   |3.8895638 |
|1     |356    |4.0   |4.8959413 |
|1     |362    |5.0   |4.492668  |
|1     |367    |4.0   |3.701784  |
|1     |423    |3.0   |3.0161366 |
|1     |441    |4.0   |4.70996   |
|1     |457    |5.0 

In [14]:
model.userFactors.show(5)

+---+--------------------+
| id|            features|
+---+--------------------+
| 10|[-0.5321485, -0.2...|
| 20|[-0.36140904, -0....|
| 30|[-0.25045606, 0.2...|
| 40|[-0.9607194, -0.7...|
| 50|[-0.45792902, -0....|
+---+--------------------+
only showing top 5 rows


In [15]:
model.itemFactors.show(5)

+---+--------------------+
| id|            features|
+---+--------------------+
| 10|[-0.18362916, -0....|
| 20|[0.20060958, -0.2...|
| 30|[-0.84820396, 0.1...|
| 40|[-0.9775326, -0.3...|
| 50|[-0.6346372, -0.1...|
+---+--------------------+
only showing top 5 rows


In [16]:
from pyspark.ml.evaluation import RegressionEvaluator

In [25]:
als = ALS (
    userCol = "userId",
    itemCol = "movieId",
    ratingCol= "rating",
    coldStartStrategy="drop"
)

(training_data, validation_data) = ratings.randomSplit([8.0, 2.0])


evaluator = RegressionEvaluator(
    metricName = "rmse",
    labelCol = "rating",
    predictionCol= "prediction"
)

#predictions = model.transform(ratings)
#rmse = evaluator.evaluate(predictions)
#print(f"Root-mean-square error = {rmse}")


model = als.fit(training_data)
predictions = model.transform(validation_data)

In [28]:
#predictions = model.transform(ratings)
#predictions.show(5)

In [29]:
rmse = evaluator.evaluate(predictions.na.drop())

In [30]:
print(rmse)

0.8732278155382708


In [31]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

parameter_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [1, 5, 10])
    .addGrid(als.maxIter, [20])
    .addGrid(als.regParam, [0.05, 0.1, 0.2])
    .build()
)

In [32]:
type(parameter_grid)

list

In [33]:
from pprint import pprint
pprint(parameter_grid)

[{Param(parent='ALS_a9d7bde4e7db', name='maxIter', doc='max number of iterations (>= 0).'): 20,
  Param(parent='ALS_a9d7bde4e7db', name='rank', doc='rank of the factorization'): 1,
  Param(parent='ALS_a9d7bde4e7db', name='regParam', doc='regularization parameter (>= 0).'): 0.05},
 {Param(parent='ALS_a9d7bde4e7db', name='maxIter', doc='max number of iterations (>= 0).'): 20,
  Param(parent='ALS_a9d7bde4e7db', name='rank', doc='rank of the factorization'): 1,
  Param(parent='ALS_a9d7bde4e7db', name='regParam', doc='regularization parameter (>= 0).'): 0.1},
 {Param(parent='ALS_a9d7bde4e7db', name='maxIter', doc='max number of iterations (>= 0).'): 20,
  Param(parent='ALS_a9d7bde4e7db', name='rank', doc='rank of the factorization'): 1,
  Param(parent='ALS_a9d7bde4e7db', name='regParam', doc='regularization parameter (>= 0).'): 0.2},
 {Param(parent='ALS_a9d7bde4e7db', name='maxIter', doc='max number of iterations (>= 0).'): 20,
  Param(parent='ALS_a9d7bde4e7db', name='rank', doc='rank of th

In [34]:
crossvalidator = CrossValidator(
    estimator= als,
    estimatorParamMaps= parameter_grid,
    evaluator= evaluator,
    numFolds= 2
)

crossval_model = crossvalidator.fit(training_data)
predictions = crossval_model.transform(validation_data)

In [36]:
rmse = evaluator.evaluate(predictions.na.drop())
print(rmse)

0.8713915402305976


In [41]:
params_dict = crossval_model.bestModel.extractParamMap()

for p, v in params_dict.items():
    print(f"{p.name}: {v}")

blockSize: 4096
predictionCol: prediction
coldStartStrategy: drop
itemCol: movieId
userCol: userId


In [42]:
best_model = crossval_model.bestModel

print(f"Mejor Rank: {best_model.rank}")
print(f"Mejor RegParam: {best_model._java_obj.parent().getRegParam()}")

Mejor Rank: 1
Mejor RegParam: 0.1


In [43]:
# Muestra el error promedio para cada combinación de la cuadrícula (ParamGrid)
print("Resultados de la métrica (ej. RMSE) para cada modelo:")
print(crossval_model.avgMetrics)

Resultados de la métrica (ej. RMSE) para cada modelo:
[np.float64(0.9524905390186928), np.float64(0.9106689856730925), np.float64(0.9232839995465374), np.float64(1.031847588093373), np.float64(0.9621184773025095), np.float64(0.9273255164600076), np.float64(1.066072544956902), np.float64(0.9677910103396955), np.float64(0.929396806706519)]


# ALS

In [7]:
from ast import With
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder


In [13]:
spark = SparkSession.builder.appName("Chapter4-5").getOrCreate()

In [14]:
ratings = (spark.read.csv(
    path="ratings.csv",
    sep=",",
    header=True,
    quote='"',
    schema="userId INT, movieId INT, rating DOUBLE, timestamp INT"
).select("userId", "movieId", "rating").cache())

In [16]:
movies = (
    spark.read.csv(
        path = "movies.csv",
        sep =",",
        header = True,
        quote = '"',
        schema = "movieId INT, title STRING, genres STRING")
.withColumn("release_year", f.regexp_extract(f.col("title"), "\s?\((\d{4})\)", 1))
.withColumn("title", f.regexp_replace(f.col("title"), "\s?\((\d{4})\)", ""))
.withColumn("genres", f.split(f.col("genres"), "\|"))
.cache()
)

<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\|'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_8755/2588459578.py:8: SyntaxWarning: invalid escape sequence '\s'
  .withColumn("release_year", f.regexp_extract(f.col("title"), "\s?\((\d{4})\)", 1))
/tmp/ipykernel_8755/2588459578.py:9: SyntaxWarning: invalid escape sequence '\s'
  .withColumn("title", f.regexp_replace(f.col("title"), "\s?\((\d{4})\)", ""))
/tmp/ipykernel_8755/2588459578.py:10: SyntaxWarning: invalid escape sequence '\|'
  .withColumn("genres", f.split(f.col("genres"), "\|"))


In [17]:
movies.show()
movies.describe().show()
ratings.describe().show()

+-------+--------------------+--------------------+------------+
|movieId|               title|              genres|release_year|
+-------+--------------------+--------------------+------------+
|      1|           Toy Story|[Adventure, Anima...|        1995|
|      2|             Jumanji|[Adventure, Child...|        1995|
|      3|    Grumpier Old Men|   [Comedy, Romance]|        1995|
|      4|   Waiting to Exhale|[Comedy, Drama, R...|        1995|
|      5|Father of the Bri...|            [Comedy]|        1995|
|      6|                Heat|[Action, Crime, T...|        1995|
|      7|             Sabrina|   [Comedy, Romance]|        1995|
|      8|        Tom and Huck|[Adventure, Child...|        1995|
|      9|        Sudden Death|            [Action]|        1995|
|     10|           GoldenEye|[Action, Adventur...|        1995|
|     11|American Presiden...|[Comedy, Drama, R...|        1995|
|     12|Dracula: Dead and...|    [Comedy, Horror]|        1995|
|     13|               B

In [19]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
)

# Determine best model parameters
evaluator = RegressionEvaluator(
    metricName="rmse", labelCol="rating", predictionCol="prediction"
)


parameter_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [1, 5, 10])
    .addGrid(als.maxIter, [20])
    .addGrid(als.regParam, [0.05, 0.1])
    .build())

crossvalidator = CrossValidator(
    estimator=als,
    estimatorParamMaps=parameter_grid,
    evaluator=evaluator,
    numFolds=2,
)

(training_data, validation_data) = ratings.randomSplit([8.0, 0.2])
crossval_model = crossvalidator.fit(training_data)

# Set model to the bestModel as determined by the CrossValidator
model = crossval_model.bestModel

# Statistics about our trained model
predictions = model.transform(validation_data).na.drop()
print(f"rmse for best model ({model}): {evaluator.evaluate(predictions)}")


rmse for best model (ALSModel: uid=ALS_ff835d2b1416, rank=1): 0.8881593899941971


In [21]:
predictions.toPandas()

,userId,movieId,rating,prediction
0,148,40815,4.0,3.589275
1,148,89745,4.0,3.647086
2,463,1552,4.5,3.127703
3,471,356,3.0,3.948212
4,31,62,4.0,3.828258
...,...,...,...,...
2450,517,1287,0.5,2.638045
2451,517,2135,3.0,2.543495
2452,517,3034,3.0,2.418052
2453,517,6373,1.5,2.223556


In [22]:
USER_ID = 150

rec_all_users = model. recommendForAllUsers(5).cache()
rec_all_users.show(20, False)

recommendations_for_user1 = (
    rec_all_users.filter(f"userId == {USER_ID}")
    # Use explode to convert the array to rows with structs
    .withColumn("rec", f.explode("recommendations"))
    # Select the columns we want from the resulting struct
    .select(
        "userId",
        f.col("rec").movieId.alias("movieId"),
        f.col("rec").rating.alias("rating"),
    )
# Join movies dataframe and select only the columns we want
    .join(movies, "movieId")
    .orderBy("rating", ascending=False)
    .select("movieId", "title", "release_year")
)

recommendations_for_user1.show(5, False)

+------+-------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                  |
+------+-------------------------------------------------------------------------------------------------+
|471   |[{6835, 7.6182804}, {5746, 7.6182804}, {40491, 7.1684985}, {136850, 7.115456}, {7899, 6.856452}] |
|463   |[{6835, 8.093259}, {5746, 8.093259}, {40491, 7.6154346}, {136850, 7.559085}, {7899, 7.2839327}]  |
|496   |[{6835, 7.245114}, {5746, 7.245114}, {40491, 6.817364}, {136850, 6.7669196}, {7899, 6.520602}]   |
|148   |[{6835, 7.466722}, {5746, 7.466722}, {40491, 7.0258884}, {136850, 6.9739013}, {7899, 6.72005}]   |
|540   |[{6835, 8.789751}, {5746, 8.789751}, {40491, 8.270806}, {136850, 8.209607}, {7899, 7.910776}]    |
|392   |[{6835, 7.3610296}, {5746, 7.3610296}, {40491, 6.926436}, {136850, 6.8751845}, {7899, 6.624926}] |
|243   |[{6835, 9.786733}, {5746, 9.7

In [24]:
USER_ID = 98

rec_all_users = model. recommendForAllUsers(5).cache().distinct()
rec_all_users.show(20, False)

recommendations_for_user1 = (
    rec_all_users.filter(f"userId == {USER_ID}")
    # Use explode to convert the array to rows with structs
    .withColumn("rec", f.explode("recommendations"))
    # Select the columns we want from the resulting struct
    .select(
        "userId",
        f.col("rec").movieId.alias("movieId"),
        f.col("rec").rating.alias("rating"),
    )
# Join movies dataframe and select only the columns we want
    .join(movies, "movieId")
    .orderBy("rating", ascending=False)
    .select("movieId", "title", "release_year")
)

recommendations_for_user1.show(5, False)

+------+-------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                  |
+------+-------------------------------------------------------------------------------------------------+
|471   |[{6835, 7.6182804}, {5746, 7.6182804}, {40491, 7.1684985}, {136850, 7.115456}, {7899, 6.856452}] |
|463   |[{6835, 8.093259}, {5746, 8.093259}, {40491, 7.6154346}, {136850, 7.559085}, {7899, 7.2839327}]  |
|496   |[{6835, 7.245114}, {5746, 7.245114}, {40491, 6.817364}, {136850, 6.7669196}, {7899, 6.520602}]   |
|148   |[{6835, 7.466722}, {5746, 7.466722}, {40491, 7.0258884}, {136850, 6.9739013}, {7899, 6.72005}]   |
|540   |[{6835, 8.789751}, {5746, 8.789751}, {40491, 8.270806}, {136850, 8.209607}, {7899, 7.910776}]    |
|243   |[{6835, 9.786733}, {5746, 9.786733}, {40491, 9.208926}, {136850, 9.140786}, {7899, 8.80806}]     |
|392   |[{6835, 7.3610296}, {5746, 7.

In [26]:
USER_ID = 150

movies_to_be_rated = (
  ratings
  # Select all movieIds that this user has not rated yet
  .filter(f"userId != {USER_ID}")
  .select("movieId").distinct()
  # Add the userId back to the data
  .withColumn("userId", f.lit(USER_ID)))

# Apply predictions
user_movie_predictions = model.transform(movies_to_be_rated)
user_movie_predictions.show(5)

+-------+------+----------+
|movieId|userId|prediction|
+-------+------+----------+
|   1580|   150|  3.594751|
|   2366|   150|  3.746734|
|   3175|   150| 3.6997511|
|   1088|   150|  3.532207|
|  32460|   150| 4.2409143|
+-------+------+----------+
only showing top 5 rows


In [27]:
# Extract recommendations
recommendations_for_user3 = (
    user_movie_predictions
    .dropna( )
    .orderBy("prediction", ascending=False)
    .limit(5)
    .join(movies, "movieId")
    .select("userId", "movieId", "title", "release_year", f.col("prediction") .alias("rating"))
)
recommendations_for_user3.show(5, False)


+------+-------+------------------------------------------------+------------+---------+
|userId|movieId|title                                           |release_year|rating   |
+------+-------+------------------------------------------------+------------+---------+
|150   |5746   |Galaxy of Terror (Quest)                        |1981        |8.196651 |
|150   |5764   |Looker                                          |1981        |7.3769855|
|150   |6835   |Alien Contamination                             |1980        |8.196651 |
|150   |40491  |Match Factory Girl, The (Tulitikkutehtaan tyttö)|1990        |7.7127223|
|150   |136850 |Villain                                         |1971        |7.6556535|
+------+-------+------------------------------------------------+------------+---------+

